# RIS CSI-feedback & beam-selection dataset — generation notebook

Works through the *Dataset Generation Guide* one step at a time. Each section writes its outputs to
`data/v1_narrowband/` so later steps can start from them.

| Section | Status |
|---|---|
| Step 1 — Geometry (BS, RIS, users, orientation) | **this notebook** |
| Step 2 — Link channels G, D, H | to do |
| Step 3 — Cascaded channel, labels, CsiNet inputs | to do |
| Step 4–6 — Variants, splits, validation | to do |

**Where to run it:** Steps 1.1–1.2 run offline. From Step 1.3 on, the notebook needs the DeepMIMO scenario
(`deepmimo==4.0.5`, data from deepmimo.net). Run it on a machine that can reach deepmimo.net.

Pure geometry helpers (frames, codebook, bin prediction) are in `code/dataset/ris_geometry.py`, so Steps 2–3 can reuse them.

In [ ]:
import sys, json, platform
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "code" / "dataset" / "ris_geometry.py").exists():
    if REPO.parent == REPO:
        raise FileNotFoundError("run the notebook from inside the repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "code" / "dataset"))
import ris_geometry as rg

DATA = REPO / "data" / "v1_narrowband"
FIGS = DATA / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
SEED = 2026
print("repo:", REPO)

## Step 1 — Geometry

Goal: choose the BS site, the RIS site and its orientation, and the user set, so users span many RIS beam
directions. Everything else stays as in RISnet.

| Element | Setting |
|---|---|
| BS | scenario TX site, 3×3 UPA, 0.5λ, ≥ 5 m high, ideally no LoS to the users |
| RIS | second TX site, 36×36 UPA, 0.25λ, 6–10 m high, boresight toward the user area |
| Anchors | rows/columns 4, 13, 22, 31 (16 elements) |
| Users | grid points 30–120 m from the RIS, 2 m high, in front of the panel |
| BS–RIS | LoS preferred |

DeepMIMO v4.0.5 conventions, checked against its source and tested in 1.2:

* Panel `shape=[My, Mz]` lies in the local y–z plane; element index `n = ny + nz·My`.
* Element response `exp(+j2π d (ny·u_y + nz·u_z))` with local direction sines `u_y = sinθ′ sinφ′`, `u_z = cosθ′`.
* Channels are `a_rx ⊗ a_tx` **with no conjugate**. So in `c = g ⊙ (H w)` both links add phase, and the best
  DFT beam sits at the bin of `u_ue + u_bs`. This lets Step 1 predict labels from geometry before any channel exists.
* `rotation=[0, downtilt, azimuth]` (degrees, downtilt > 0 = below horizon) aims the boresight (local +x).
  `Dataset.bs_look_at()` in 4.0.5 writes `[az, el, 0]` into the `[x, y, z]` slots and does **not** aim the array, so we don't use it.

### 1.1 Fixed array parameters and what the codebook can resolve (offline)

In [ ]:
D_ris = rg.aperture(rg.RIS_SHAPE[0], rg.RIS_SPACING)
rows = {
    "carrier [GHz]": rg.FC / 1e9,
    "wavelength [mm]": 1e3 * rg.LAMBDA,
    "RIS aperture per side [m]": D_ris,
    "far-field distance, side [m]": rg.far_field_distance(D_ris),
    "far-field distance, diagonal [m]": rg.far_field_distance(np.sqrt(2) * D_ris),
    "DFT bin spacing in direction sine": 1 / (rg.RIS_SHAPE[0] * rg.RIS_SPACING),
    "visible bins per axis (|u| <= 1), 36-pt": int(np.sum(np.abs(np.arange(-18, 18)) <= 36 * rg.RIS_SPACING)),
    "visible bins per axis (|u| <= 1), 72-pt": int(np.sum(np.abs(np.arange(-36, 36)) <= 72 * rg.RIS_SPACING)),
    "BS aperture per side [m]": rg.aperture(rg.BS_SHAPE[0], rg.BS_SPACING),
}
for k, v in rows.items():
    print(f"{k:45s} {v:10.3f}")
ANCHOR_IDX = rg.anchor_indices()
print("anchor element indices:", ANCHOR_IDX)

**Feasibility of the label target.** The users stand at 2 m and the RIS at 6–10 m, so between 30 and 120 m they
span only a few degrees of elevation: `u_z` covers about 0.1–0.2, which is 1–2 DFT bins of 0.111. Label diversity therefore
comes almost entirely from azimuth (at most 19 bins). The cell below runs an idealised user area (a full half-annulus in
front of the RIS) through the label predictor. It gives an upper bound for any real street layout.

In [ ]:
xs = np.arange(-130, 130, 0.5)
X, Y = np.meshgrid(xs, xs)
P = np.c_[X.ravel(), Y.ravel(), np.full(X.size, rg.UE_HEIGHT)]
bs_dummy = np.array([-40.0, 60.0, 6.0])     # incident direction only shifts the bins
res = []
for h in (6, 8, 10):
    ris = np.array([0.0, 0.0, h])
    for d_min in (15, 30):
        for span in (90, 180):
            d = np.linalg.norm(P - ris, axis=1)
            az = np.degrees(np.arctan2(P[:, 1], P[:, 0]))
            m = (d >= d_min) & (d <= rg.D_MAX) & (np.abs(az) <= span / 2)
            rot = rg.look_at_rotation(ris, P[m].mean(0))
            row = [h, d_min, span]
            for ov in (1, 2):
                lab, *_ = rg.predict_labels(P[m], ris, bs_dummy, rot, ov)
                row.append(rg.label_stats(lab, 50)["n_ge_min"])
            res.append(row)
print("RIS h | d_min | azimuth span | labels>=50 (36x36) | labels>=50 (72x72)")
for r in res:
    print(f"{r[0]:5.0f} | {r[1]:5.0f} | {r[2]:12.0f} | {r[3]:18d} | {r[4]:18d}")

Even this ideal layout gives only about 30–45 labels with ≥ 50 samples for the 36×36 codebook at `d_min = 30 m`,
and about 80–150 for the 2× oversampled 72×72 codebook. **The "≥ 100 labels" target is not reachable with the 36×36 codebook
under the Step 1 constraints.** Options (decide before Step 3):

1. Use the 72×72 oversampled codebook as the main label space (recommended: it keeps the far-field geometry). Report 36×36 as an ablation.
2. Keep 36×36 and relax the target to about 40 labels.
3. Lower `d_min` toward 15 m (above the 14 m side far-field distance but below the 28 m diagonal one) and state the far-field assumption.

The rest of Step 1 reports both codebooks for the real scenario.

### 1.2 Self-test: element ordering, rotation and bin prediction (offline)

Checklist item *"a steering vector toward a known position peaks at the DFT bin predicted from geometry"*. We use
DeepMIMO's own internal array-response and rotation functions, so a convention change in a future version is caught here.

In [ ]:
import deepmimo as dm
from deepmimo.generator.geometry import _rotate_angles, _ant_indices, _array_response
print("deepmimo", dm.__version__)

Psi1 = rg.ris_codebook(oversample=1)
Psi2 = rg.ris_codebook(oversample=2)
assert np.allclose(Psi1.conj().T @ Psi1, np.eye(Psi1.shape[1]))

rng = np.random.default_rng(SEED)
ris = np.array([10.0, 20.0, 8.0])
ant = _ant_indices(rg.RIS_SHAPE)
max_err, hits1, hits2, n_trials = 0.0, 0, 0, 300
for _ in range(n_trials):
    rot = rg.look_at_rotation(ris, ris + rng.uniform([-60, -60, -6], [60, 60, -5]))
    tgt = ris + rng.uniform([-100, -100, -6], [100, 100, -5])
    d = rg.unit(tgt - ris)
    th, ph = np.degrees(np.arccos(d[2])), np.degrees(np.arctan2(d[1], d[0]))
    t2, p2 = _rotate_angles(rot, th, ph)                       # DeepMIMO's rotation (radians out)
    loc_dm = np.array([np.sin(t2) * np.cos(p2), np.sin(t2) * np.sin(p2), np.cos(t2)])
    loc = rg.to_local(d, rot)
    max_err = max(max_err, np.abs(loc_dm - loc).max())
    a = _array_response(ant, t2, p2, 2 * np.pi * rg.RIS_SPACING).ravel()
    hits1 += np.argmax(np.abs(a @ Psi1)) == rg.predicted_beam(loc[1], loc[2], oversample=1)
    hits2 += np.argmax(np.abs(a @ Psi2)) == rg.predicted_beam(loc[1], loc[2], oversample=2)

# look_at_rotation puts the target exactly on boresight
tgt = np.array([60.0, 70.0, 2.0])
bore = rg.direction_sines(ris, tgt[None], rg.look_at_rotation(ris, tgt))[0]
print(f"frame error vs DeepMIMO: {max_err:.1e}")
print(f"predicted bin == argmax |a^T Psi|: 36x36 {hits1}/{n_trials}, 72x72 {hits2}/{n_trials}")
print("boresight direction in local frame:", np.round(bore, 12))
assert max_err < 1e-9 and hits1 == n_trials and hits2 == n_trials and np.allclose(bore, [1, 0, 0])

### 1.3 Scenario inventory (needs the scenario)

Set `SCEN` after checking `dm.search(...)` (Step 0). TX sets are the candidate BS/RIS sites. RX-only sets with more
than one point are the user grids.

In [ ]:
SCEN = "o1_3p5"          # confirm with dm.search({'bands': ['sub6'], 'environment': 'outdoor'})
MAX_PATHS_SCREEN = 1     # screening needs only positions and the LoS flag

if not Path(dm.get_scenario_folder(SCEN)).exists():
    dm.download(SCEN)
dm.summary(SCEN)          # paste into the guide's "Open questions"

SETS = dm.get_txrx_sets(SCEN)
for s in SETS:
    print(s)
TX_SITES = [(s.id, i) for s in SETS if s.is_tx for i in range(s.num_points)]
UE_SETS = [s.id for s in SETS if s.is_rx and not s.is_tx and s.num_points > 1]
BS2BS_SETS = {s.id for s in SETS if s.is_tx and s.is_rx}
print(f"\n{len(TX_SITES)} TX sites; user RX sets {UE_SETS}; TX sets that are also RX (BS-to-BS): {sorted(BS2BS_SETS)}")

In [ ]:
def _parts(ds):
    return list(ds.datasets) if isinstance(ds, dm.MacroDataset) else [ds]

def load_link(tx_site, rx_sets, matrices=("rx_pos", "tx_pos", "inter", "aoa_az"), max_paths=MAX_PATHS_SCREEN):
    # Load one TX site toward RX sets. rx_sets: list of ids ('all' points) or dict {id: idx list}.
    tx_set, tx_idx = tx_site
    rx = rx_sets if isinstance(rx_sets, dict) else {r: "all" for r in rx_sets}
    ds = dm.load(SCEN, max_paths=max_paths, tx_sets={tx_set: [tx_idx]}, rx_sets=rx, matrices=list(matrices))
    parts = _parts(ds)
    out = {
        "tx_pos": np.asarray(parts[0].tx_pos, float).reshape(-1, 3)[0],
        "rx_pos": np.concatenate([np.asarray(p.rx_pos, float) for p in parts]),
        "rx_set": np.concatenate([np.full(len(p.rx_pos), p.txrx["rx_set_id"]) for p in parts]),
        "rx_idx": np.concatenate([np.asarray(rx[p.txrx["rx_set_id"]]) if not isinstance(rx[p.txrx["rx_set_id"]], str)
                                  else np.arange(len(p.rx_pos)) for p in parts]),
    }
    if "inter" in matrices:
        out["los"] = np.concatenate([np.asarray(p.los) for p in parts]).astype(np.int8)
    for k in ("aod_el", "aod_az"):
        if k in matrices:
            out[k] = np.concatenate([np.asarray(p[k], float) for p in parts])
    return out

### 1.4 Screen every TX site as an RIS candidate

For each site: aim the boresight at the centroid of users 30–120 m away (iterated with the in-front mask), then count users
and LoS share, and predict labels from user directions alone. Every site's LoS map toward the users is kept, because in 1.5
the same site can serve as the BS.

In [ ]:
links, screen = {}, []
for site in TX_SITES:
    L = load_link(site, UE_SETS)
    on_grid = np.abs(L["rx_pos"][:, 2] - rg.UE_HEIGHT) < 0.25
    L["los"][~on_grid] = -1                       # ignore non-ground RX points
    links[site] = L
    rot, m = rg.aim_ris(L["rx_pos"], L["tx_pos"], L["los"])
    row = {"site": site, "pos": L["tx_pos"], "height": L["tx_pos"][2], "n_users": int(m.sum())}
    if rot is not None and m.sum() > 0:
        u = rg.direction_sines(L["tx_pos"], L["rx_pos"][m], rot)
        for ov in (1, 2):
            lab = rg.predicted_beam(u[:, 1], u[:, 2], oversample=ov)
            row[f"labels{ov}x"] = rg.label_stats(lab)["n_ge_min"]
        row.update(rot=rot, los_share=float((L["los"][m] == 1).mean()),
                   az_span_deg=float(np.ptp(np.degrees(np.arctan2(u[:, 1], u[:, 0])))))
    screen.append(row)

screen.sort(key=lambda r: (r.get("labels2x", 0), r.get("labels1x", 0)), reverse=True)
print(" site     | height | users  | LoS share | az span | labels>=50 36x36 | 72x72")
for r in screen:
    if "rot" in r:
        print(f" {str(r['site']):8s} | {r['height']:6.1f} | {r['n_users']:6d} | {r['los_share']:9.2f} | "
              f"{r['az_span_deg']:7.1f} | {r['labels1x']:16d} | {r['labels2x']:5d}")
    else:
        print(f" {str(r['site']):8s} | {r['height']:6.1f} | {r['n_users']:6d} | (no users in window)")

### 1.5 Pick the (RIS, BS) pair

For the best RIS candidates, try every other site as the BS. A BS must be ≥ 5 m high and in front of the RIS panel.
The ranking prefers, in order: a BS→RIS LoS path (BS-to-BS data), more labels with the incident direction included
(72×72, then 36×36), and a lower share of users with LoS from the BS.
Set `RIS_SITE` / `BS_SITE` by hand to override the automatic choice.

In [ ]:
TOP_RIS = 4
RIS_SITE, BS_SITE = None, None      # e.g. (1, 4), (1, 2) to override

def bs_to_ris_los(bs_site, ris_site):
    # 1 = LoS, 0 = NLoS, -1 = no path, None = scenario has no BS-to-BS data for this pair
    if ris_site[0] not in BS2BS_SETS:
        return None
    try:
        L = load_link(bs_site, {ris_site[0]: [ris_site[1]]}, matrices=("rx_pos", "tx_pos", "inter", "aoa_az"))
        return int(L["los"][0])
    except Exception as e:                     # missing BS-to-BS files
        print("BS-to-BS load failed", bs_site, "->", ris_site, ":", e)
        return None

pairs = []
for r in [r for r in screen if "rot" in r][:TOP_RIS]:
    ris_site, Lr = r["site"], links[r["site"]]
    m, _ = rg.select_users(Lr["rx_pos"], Lr["tx_pos"], r["rot"], Lr["los"])
    for bs_site in TX_SITES:
        if bs_site == ris_site:
            continue
        bs_pos = links[bs_site]["tx_pos"]
        u_bs = rg.direction_sines(Lr["tx_pos"], bs_pos[None], r["rot"])[0]
        if bs_pos[2] < 5 or u_bs[0] <= 0:
            continue
        p = {"ris": ris_site, "bs": bs_site, "d_bs_ris": float(np.linalg.norm(bs_pos - Lr["tx_pos"])),
             "bs_ue_los_share": float((links[bs_site]["los"][m] == 1).mean()),
             "bs_ris_los": bs_to_ris_los(bs_site, ris_site)}
        for ov in (1, 2):
            lab, *_ = rg.predict_labels(Lr["rx_pos"][m], Lr["tx_pos"], bs_pos, r["rot"], ov)
            p[f"labels{ov}x"] = rg.label_stats(lab)["n_ge_min"]
        pairs.append(p)

pairs.sort(key=lambda p: (p["bs_ris_los"] == 1, p["labels2x"], p["labels1x"], -p["bs_ue_los_share"]), reverse=True)
print(" RIS      | BS       | d(BS,RIS) | BS-RIS LoS | BS->UE LoS share | labels 36x36 | 72x72")
for p in pairs[:15]:
    print(f" {str(p['ris']):8s} | {str(p['bs']):8s} | {p['d_bs_ris']:9.1f} | {str(p['bs_ris_los']):10s} | "
          f"{p['bs_ue_los_share']:16.2f} | {p['labels1x']:12d} | {p['labels2x']:5d}")
if not pairs:
    raise RuntimeError("no valid (RIS, BS) pair: relax the height / in-front constraints")
if RIS_SITE is None:
    RIS_SITE, BS_SITE = pairs[0]["ris"], pairs[0]["bs"]
print("\nchosen: RIS", RIS_SITE, " BS", BS_SITE)

### 1.6 Final geometry, user set and checks

`GRID_STEP_M` thins the 0.2 m grid to keep about `TARGET_USERS` points. Neighbouring 0.2 m points are near-duplicates anyway;
Step 5 splits by 5 × 5 m blocks.

In [ ]:
TARGET_USERS = 60_000
GRID_STEP_M = None        # None = smallest of 0.2, 0.4, ... m that keeps <= TARGET_USERS

Lr, Lb = links[RIS_SITE], links[BS_SITE]
RIS_POS, BS_POS = Lr["tx_pos"], Lb["tx_pos"]
RIS_ROT = next(r["rot"] for r in screen if r["site"] == RIS_SITE)
m_win, dist = rg.select_users(Lr["rx_pos"], RIS_POS, RIS_ROT, Lr["los"])

if GRID_STEP_M is None:
    for step in np.round(np.arange(0.2, 5.01, 0.2), 1):
        keep = m_win & rg.decimate_grid(Lr["rx_pos"], step if step > 0.2 else 0)
        if keep.sum() <= TARGET_USERS:
            GRID_STEP_M = float(step)
            break
keep = m_win & rg.decimate_grid(Lr["rx_pos"], GRID_STEP_M if GRID_STEP_M > 0.2 else 0)
sel = np.flatnonzero(keep)

UE_POS = Lr["rx_pos"][sel]
lab1, u_ue, u_bs = rg.predict_labels(UE_POS, RIS_POS, BS_POS, RIS_ROT, 1)
lab2, *_ = rg.predict_labels(UE_POS, RIS_POS, BS_POS, RIS_ROT, 2)
st1, st2 = rg.label_stats(lab1), rg.label_stats(lab2)
bs_ris_los = bs_to_ris_los(BS_SITE, RIS_SITE)
los_ris, los_bs = Lr["los"][sel], Lb["los"][sel]

checks = [
    ("RIS height 6-10 m", 6 <= RIS_POS[2] <= 10, f"{RIS_POS[2]:.1f} m"),
    ("BS height >= 5 m", BS_POS[2] >= 5, f"{BS_POS[2]:.1f} m"),
    ("BS in front of RIS", u_bs[0] > 0, f"boresight cos = {u_bs[0]:.2f}"),
    ("BS-RIS LoS", bs_ris_los == 1, f"{bs_ris_los} (None = no BS-to-BS data, use reciprocity fallback)"),
    ("users 30-120 m (far field)", dist[sel].min() >= rg.D_MIN, f"{dist[sel].min():.1f}-{dist[sel].max():.1f} m"),
    ("all users in front of RIS", (u_ue[:, 0] > 0).all(), f"min cos = {u_ue[:, 0].min():.2f}"),
    ("BS has little LoS to users", (los_bs == 1).mean() < 0.2, f"BS->UE LoS share {(los_bs == 1).mean():.2f}"),
    (">=100 labels with >=50 samples, 36x36", st1["n_ge_min"] >= 100, f"{st1['n_ge_min']} (distinct {st1['n_distinct']})"),
    (">=100 labels with >=50 samples, 72x72", st2["n_ge_min"] >= 100, f"{st2['n_ge_min']} (distinct {st2['n_distinct']})"),
]
print(f"RIS {RIS_SITE} at {np.round(RIS_POS, 2)}, rotation [0, tilt, az] = {np.round(RIS_ROT, 2)} deg")
print(f"BS  {BS_SITE} at {np.round(BS_POS, 2)}, |BS-RIS| = {np.linalg.norm(BS_POS - RIS_POS):.1f} m")
print(f"users: {len(sel)} (grid step {GRID_STEP_M} m, {m_win.sum()} before thinning), RIS->UE LoS share {(los_ris == 1).mean():.2f}")
print(f"azimuth span seen from RIS: {np.ptp(np.degrees(np.arctan2(u_ue[:, 1], u_ue[:, 0]))):.1f} deg, "
      f"u_z range {u_ue[:, 2].min():.3f}..{u_ue[:, 2].max():.3f}\n")
for name, ok, info in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name:42s} {info}")

**Cross-check on real data:** for users with LoS from the RIS, the stored departure angles of the LoS path must match the
geometric direction. This confirms that `aod_el` is a zenith angle and that azimuth is measured from +x, so the
predicted labels transfer to the ray-traced channels in Step 3.

In [ ]:
los_sel = sel[los_ris == 1]
if len(los_sel):
    chk = rng.choice(los_sel, size=min(2000, len(los_sel)), replace=False)
    sets_ = {s: Lr["rx_idx"][chk[Lr["rx_set"][chk] == s]].tolist() for s in np.unique(Lr["rx_set"][chk])}
    A = load_link(RIS_SITE, {int(k): v for k, v in sets_.items()},
                  matrices=("rx_pos", "tx_pos", "inter", "aoa_az", "aod_el", "aod_az"), max_paths=1)
    d = rg.unit(A["rx_pos"] - RIS_POS)
    th_geo = np.degrees(np.arccos(d[:, 2]))
    ph_geo = np.degrees(np.arctan2(d[:, 1], d[:, 0]))
    ok = A["los"] == 1
    e_th = np.abs(A["aod_el"][ok, 0] - th_geo[ok])
    e_ph = np.abs((A["aod_az"][ok, 0] - ph_geo[ok] + 180) % 360 - 180)
    print(f"LoS angle error over {ok.sum()} users: zenith median {np.median(e_th):.3f} / max {e_th.max():.3f} deg, "
          f"azimuth median {np.median(e_ph):.3f} / max {e_ph.max():.3f} deg")
    assert np.median(e_th) < 0.5 and np.median(e_ph) < 0.5, "angle convention mismatch: check before Step 2"
else:
    print("no LoS users from the RIS: skip angle cross-check")

### 1.7 Figures

Beam map (checklist item): predicted labels should form smooth, contiguous regions.

In [ ]:
allp = links[RIS_SITE]["rx_pos"]
grid = allp[np.abs(allp[:, 2] - rg.UE_HEIGHT) < 0.25]
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, lab, title in ((ax[0], lab1, "36x36"), (ax[1], lab2, "72x72")):
    a.scatter(grid[::20, 0], grid[::20, 1], s=0.2, c="0.85", rasterized=True)
    _, inv = np.unique(lab, return_inverse=True)
    perm = np.random.default_rng(0).permutation(inv.max() + 1)          # shuffled colours separate neighbours
    a.scatter(UE_POS[:, 0], UE_POS[:, 1], s=0.5, c=perm[inv], cmap="tab20", rasterized=True)
    for site in TX_SITES:
        p = links[site]["tx_pos"]
        a.plot(*p[:2], "k^", ms=4)
        a.annotate(str(site), p[:2], fontsize=6)
    a.plot(*RIS_POS[:2], "rs", ms=9, label="RIS")
    a.plot(*BS_POS[:2], "b^", ms=9, label="BS")
    az = np.radians(RIS_ROT[2])
    a.arrow(*RIS_POS[:2], 25 * np.cos(az), 25 * np.sin(az), color="r", width=0.8)
    a.set_aspect("equal"); a.legend(loc="upper right")
    a.set_title(f"Predicted best beam, {title}: {len(np.unique(lab))} distinct")
    a.set_xlabel("x [m]"); a.set_ylabel("y [m]")
fig.tight_layout(); fig.savefig(FIGS / "step1_beam_map.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].hist(dist[sel], bins=60); ax[0].set_xlabel("distance to RIS [m]"); ax[0].set_ylabel("users")
for a, lab, title in ((ax[1], lab1, "36x36"), (ax[2], lab2, "72x72")):
    cnt = np.sort(np.unique(lab, return_counts=True)[1])[::-1]
    a.bar(np.arange(len(cnt)), cnt, width=1.0); a.axhline(50, color="r", ls="--", lw=1)
    a.set_yscale("log"); a.set_xlabel("label rank"); a.set_ylabel("users"); a.set_title(f"label counts {title}")
fig.tight_layout(); fig.savefig(FIGS / "step1_label_counts.png", dpi=150); plt.show()

### 1.8 Save Step 1 outputs

* `data/v1_narrowband/geometry.json`: site ids, positions, RIS rotation, array settings, user window, checks.
  Step 2 reads this.
* `data/v1_narrowband/users_step1.npz`: selected users (`rx_set`, `rx_idx`, `pos`, distances, LoS flags, predicted labels).

In [ ]:
geometry = {
    "scenario": SCEN, "deepmimo_version": dm.__version__, "python": platform.python_version(), "seed": SEED,
    "fc_hz": rg.FC,
    "bs": {"site": list(map(int, BS_SITE)), "pos": BS_POS.tolist(), "shape": list(rg.BS_SHAPE), "spacing_wl": rg.BS_SPACING,
           "rotation_deg": [0.0, 0.0, 0.0]},
    "ris": {"site": list(map(int, RIS_SITE)), "pos": RIS_POS.tolist(), "shape": list(rg.RIS_SHAPE),
            "spacing_wl": rg.RIS_SPACING, "rotation_deg": RIS_ROT.tolist(), "anchor_rows": list(rg.ANCHOR_ROWS),
            "anchor_idx": ANCHOR_IDX.tolist()},
    "users": {"rx_sets": [int(s) for s in UE_SETS], "d_min_m": rg.D_MIN, "d_max_m": rg.D_MAX, "height_m": rg.UE_HEIGHT,
              "grid_step_m": GRID_STEP_M, "n_users": int(len(sel)), "n_in_window": int(m_win.sum())},
    "bs_ris_los": bs_ris_los,
    "predicted_labels": {"36x36": st1, "72x72": st2},
    "checks": {name: bool(ok) for name, ok, _ in checks},
    "element_index": "n = ny + nz*My (DeepMIMO v4); codebook column = kz*Ky + ky",
}
(DATA / "geometry.json").write_text(json.dumps(geometry, indent=2))
np.savez_compressed(DATA / "users_step1.npz",
                    rx_set=Lr["rx_set"][sel], rx_idx=Lr["rx_idx"][sel], pos=UE_POS.astype(np.float32),
                    dist_ris=dist[sel].astype(np.float32), los_ris=los_ris, los_bs=los_bs,
                    label_pred_36=lab1, label_pred_72=lab2)
print(json.dumps(geometry, indent=2)[:1500])